# Traffic-signal-control experiment

This notebook compares simple baselines, MAPPO, and reward-sharing MAPPO on a network.

In [1]:
import os
import subprocess
import sys
import sysconfig
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/marl-tsc")

    if not ROOT.exists():
        subprocess.run([
            "git", "clone", "--branch", "neighbour-gifting",
            "--single-branch", "https://github.com/abergh18/marl-tsc.git",
            str(ROOT)
        ], check=True)

    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "eclipse-sumo"
    ], check=True)

    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(ROOT / "requirements.txt"),
    ], check=True)

    import sumo
    os.environ["SUMO_HOME"] = sumo.SUMO_HOME
    sys.path.insert(0, os.path.join(sumo.SUMO_HOME, "tools"))

else:
    # Run VS Code from somewhere inside the marl-tsc repository.
    ROOT = next(
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "marl_tsc").exists()
    )

# Make /src/marl_tsc importable.
sys.path.insert(0, str(ROOT / "src"))

# Relative project paths now work consistently.
os.chdir(ROOT)

print(f"Running in {'Colab' if IN_COLAB else 'VS Code'} from {ROOT}")

Running in Colab from /content/marl-tsc


In [2]:
if not IN_COLAB:
  %reload_ext autoreload
  %autoreload 2

import matplotlib.pyplot as plt

from marl_tsc.baselines import fixed_time_actions, random_actions
from marl_tsc.mappo import train_mappo
from marl_tsc.network_types import CityNetwork
from marl_tsc.simulation_generator import SimulationGenerator
from marl_tsc.training import (
    evaluate_policies,
    export_policy_replay,
    evaluation_results_table,
    plot_training_histories,
)

## 1. Configure and generate the simulation

In [3]:
import urllib.request

if IN_COLAB:
    # Create a custom opener to disguise script as a web browser
    opener = urllib.request.build_opener()
    opener.addheaders = [
        ("User-Agent", "Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
    ]

    # Apply this rule globally for this notebook
    urllib.request.install_opener(opener)

In [4]:
SEED = 42
EPISODE_STEPS = 600
SECONDS_PER_ACTION = 5
SIMULATION_DURATION = EPISODE_STEPS * SECONDS_PER_ACTION
TRAFFIC_SPAWN_DURATION = int(SIMULATION_DURATION * 1.20)
TOTAL_TIMESTEPS = 120_000
EVALUATION_EPISODES = 3
MIN_GREEN_SECONDS = 10

OUTPUT_DIR = ROOT / "outputs"
SIMULATION_DIR = OUTPUT_DIR / "simulation"

lancaster_network = CityNetwork(city_name="Lancaster, UK", radius=500)
generator = SimulationGenerator(
    output_dir=SIMULATION_DIR,
    network=lancaster_network,
    trip_begin=0,
    trip_end=TRAFFIC_SPAWN_DURATION,
    trip_period=3,
    seed=SEED,
)

lancaster_paths = generator.generate_all()
traffic_light_ids = list(lancaster_paths.traffic_light_ids)
print(f"Generated SUMO config: {lancaster_paths.config_file}")
print(f"Traffic-light agent count: {len(traffic_light_ids)}")

bristol_network = CityNetwork(city_name="Bristol, UK", radius=500)
BRISTOL_SIMULATION_DIR = OUTPUT_DIR / "simulation_bristol"
bristol_generator = SimulationGenerator(
    output_dir=BRISTOL_SIMULATION_DIR,
    network=bristol_network,
    trip_begin=0,
    trip_end=TRAFFIC_SPAWN_DURATION,
    trip_period=3,
    seed=SEED,
)

bristol_paths = bristol_generator.generate_all()
bristol_traffic_light_ids = list(bristol_paths.traffic_light_ids)
print(f"Bristol SUMO config: {bristol_paths.config_file}")
print(f"Bristol traffic-light agent count: {len(bristol_traffic_light_ids)}")

from marl_tsc.traffic_env import SumoTrafficEnv

env = SumoTrafficEnv(config_file=lancaster_paths.config_file, possible_agents=traffic_light_ids, max_steps=10)
obs, _ = env.reset()
SHARED_MAX_LANES = env.max_lanes_per_tls
SHARED_GREEN_PHASES = env.green_phase_count
env.close()

env = SumoTrafficEnv(config_file=bristol_paths.config_file, possible_agents=bristol_traffic_light_ids, max_steps=10)
obs, _ = env.reset()
SHARED_MAX_LANES = max(SHARED_MAX_LANES, env.max_lanes_per_tls)
SHARED_GREEN_PHASES = max(SHARED_GREEN_PHASES, env.green_phase_count)
env.close()

print(f"Shared max_lanes_per_tls: {SHARED_MAX_LANES}")
print(f"Shared green_phase_count: {SHARED_GREEN_PHASES}")

TRAIN_ENV_KWARGS = {
    "min_green_seconds": MIN_GREEN_SECONDS,
    "max_lanes_per_tls": SHARED_MAX_LANES,
    "green_phase_count": SHARED_GREEN_PHASES,
}
EVAL_ENV_KWARGS = {
    "min_green_seconds": MIN_GREEN_SECONDS,
    "seconds_per_action": SECONDS_PER_ACTION,
    "switch_penalty": 0.1,
    "max_lanes_per_tls": SHARED_MAX_LANES,
    "green_phase_count": SHARED_GREEN_PHASES,
    "collect_global_metrics": True,
    "global_metric_interval": 10,
}

Generated SUMO config: /content/marl-tsc/outputs/simulation/config.sumocfg
Traffic-light agent count: 5
Bristol SUMO config: /content/marl-tsc/outputs/simulation_bristol/config.sumocfg
Bristol traffic-light agent count: 4
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
Shared max_lanes_per_tls: 6
Shared green_phase_count: 4


## 2. Train MAPPO policies

The reward-sharing policy has an extra sharing action during training. Both policies are evaluated using the same traffic metrics.

In [ ]:
mappo_model, mappo_history, mappo_model_path = train_mappo(
    config_file=lancaster_paths.config_file,
    traffic_light_ids=traffic_light_ids,
    output_dir=(OUTPUT_DIR, 'mappo'),
    total_timesteps=TOTAL_TIMESTEPS,
    rollout_steps=256,
    max_steps=EPISODE_STEPS,
    seed=SEED,
    env_kwargs=TRAIN_ENV_KWARGS,
    use_peer_reward=False,
)

 Retrying in 1 seconds
[MAPPO] Episode 1 | Return: 5756.17 | Total Steps: 600
 Retrying in 1 seconds
[MAPPO] Episode 2 | Return: 5735.24 | Total Steps: 1200
 Retrying in 1 seconds
[MAPPO] Episode 3 | Return: 5782.85 | Total Steps: 1800
 Retrying in 1 seconds
[MAPPO] Episode 4 | Return: 5763.55 | Total Steps: 2400
 Retrying in 1 seconds
[MAPPO] Episode 5 | Return: 5684.36 | Total Steps: 3000
 Retrying in 1 seconds
[MAPPO] Episode 6 | Return: 5804.07 | Total Steps: 3600
 Retrying in 1 seconds
[MAPPO] Episode 7 | Return: 5784.05 | Total Steps: 4200
 Retrying in 1 seconds
[MAPPO] Episode 8 | Return: 5765.33 | Total Steps: 4800
 Retrying in 1 seconds
[MAPPO] Episode 9 | Return: 5833.61 | Total Steps: 5400
 Retrying in 1 seconds
[MAPPO] Episode 10 | Return: 5775.40 | Total Steps: 6000
 Retrying in 1 seconds
[MAPPO] Episode 11 | Return: 5830.69 | Total Steps: 6600
 Retrying in 1 seconds
[MAPPO] Episode 12 | Return: 5825.99 | Total Steps: 7200
 Retrying in 1 seconds
[MAPPO] Episode 13 | Return

In [ ]:
reward_sharing_model, reward_sharing_history, reward_sharing_model_path = train_mappo(
    config_file=lancaster_paths.config_file,
    traffic_light_ids=traffic_light_ids,
    output_dir=(OUTPUT_DIR, 'rs_mappo'),
    total_timesteps=TOTAL_TIMESTEPS,
    rollout_steps=256,
    max_steps=EPISODE_STEPS,
    seed=SEED,
    env_kwargs=TRAIN_ENV_KWARGS,
    use_peer_reward=True,
)

In [ ]:
fig, ax = plot_training_histories({
    "MAPPO": mappo_history,
    "Reward-sharing MAPPO": reward_sharing_history,
})
plt.show()

## 3. Export a SUMO replay

In [ ]:
replay_config = export_policy_replay(
    config_file=lancaster_paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policy=reward_sharing_model,
    output_dir=OUTPUT_DIR / "replays" / "reward_sharing_mappo",
    max_steps=EPISODE_STEPS,
    seed=SEED,
    env_kwargs=EVAL_ENV_KWARGS,
)
print(f"Open this file in the SUMO GUI: {replay_config}")

## 4. Compare all policies

Every policy uses the same evaluation episodes, seeds, and environment settings.

In [ ]:
policies = {
    "Random": random_actions,
    "Fixed-Time": fixed_time_actions,
    "MAPPO": mappo_model,
    "Reward-sharing MAPPO": reward_sharing_model,
}

policy_results_lancaster = evaluate_policies(
    config_file=lancaster_paths.config_file,
    traffic_light_ids=traffic_light_ids,
    policies=policies,
    episodes=EVALUATION_EPISODES,
    max_steps=EPISODE_STEPS,
    seed=SEED,
    env_kwargs=EVAL_ENV_KWARGS,
)

print("=== Lancaster ===")
display(evaluation_results_table(policy_results_lancaster))

policy_results_bristol = evaluate_policies(
    config_file=bristol_paths.config_file,
    traffic_light_ids=bristol_traffic_light_ids,
    policies=policies,
    episodes=EVALUATION_EPISODES,
    max_steps=EPISODE_STEPS,
    seed=SEED,
    env_kwargs=EVAL_ENV_KWARGS,
)

print("=== Bristol ===")
display(evaluation_results_table(policy_results_bristol))